# Boosting + Other Tree-Based Baselines (MFCC Summary Features)

Compares a few tree-based models using the same session split (1-4 train, 5 test)
and the MFCC summary features. Gender is excluded.

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score

repo_root = Path.cwd().parents[1]
csv_path = repo_root / "extracted_features" / "mfcc" / "mfcc_features.csv"
csv_path

WindowsPath('f:/Speech-Emotion-Recognition/extracted_features/mfcc/mfcc_features.csv')

In [7]:
df = pd.read_csv(csv_path)

# Safety filter: valid labels only
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 2)].copy()
df.shape

(3066, 1157)

In [8]:
# Define metadata columns (drop gender explicitly)
metadata_cols = [
    "path",
    "session",
    "method",
    "gender",
    "emotion",
    "n_annotators",
    "agreement",
]

feature_cols = [c for c in df.columns if c not in metadata_cols]
X = df[feature_cols].copy()
y = df["emotion"].copy()

# Drop rows with any missing values in features
mask = X.notna().all(axis=1)
X = X.loc[mask]
y = y.loc[mask]
df = df.loc[mask]

X.shape, y.shape

((3066, 1150), (3066,))

In [9]:
# Session-based split: Sessions 1-4 train, Session 5 test
train_mask = df["session"].isin([1, 2, 3, 4])
test_mask = df["session"].isin([5])

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]

X_train.shape, X_test.shape

((2522, 1150), (544, 1150))

In [10]:
def evaluate_model(name, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")

    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1:  {f1:.4f}")
    print("\nClassification report:\n")
    print(classification_report(y_test, y_pred))
    print("\n")


models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    ),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}

for name, model in models.items():
    evaluate_model(name, model)

=== RandomForest ===
Accuracy: 0.4706
Macro F1:  0.3364

Classification report:

              precision    recall  f1-score   support

         ang       0.55      0.82      0.65        65
         exc       0.67      0.06      0.12        93
         fru       0.34      0.60      0.43       133
         hap       0.00      0.00      0.00        36
         neu       0.47      0.46      0.46       117
         sad       0.72      0.65      0.68        97
         sur       0.00      0.00      0.00         3

    accuracy                           0.47       544
   macro avg       0.39      0.37      0.34       544
weighted avg       0.49      0.47      0.43       544





f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

=== GradientBoosting ===
Accuracy: 0.4540
Macro F1:  0.3039

Classification report:

              precision    recall  f1-score   support

         ang       0.51      0.82      0.63        65
         exc       0.43      0.16      0.23        93
         fea       0.00      0.00      0.00         0
         fru       0.34      0.46      0.39       133
         hap       0.00      0.00      0.00        36
         neu       0.46      0.51      0.49       117
         sad       0.82      0.60      0.69        97
         sur       0.00      0.00      0.00         3

    accuracy                           0.45       544
   macro avg       0.32      0.32      0.30       544
weighted avg       0.46      0.45      0.44       544





f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


=== AdaBoost ===
Accuracy: 0.3732
Macro F1:  0.2565

Classification report:

              precision    recall  f1-score   support

         ang       0.42      0.69      0.53        65
         exc       0.25      0.01      0.02        93
         fru       0.31      0.59      0.40       133
         hap       0.00      0.00      0.00        36
         neu       0.32      0.32      0.32       117
         sad       0.69      0.42      0.53        97
         sur       0.00      0.00      0.00         3

    accuracy                           0.37       544
   macro avg       0.28      0.29      0.26       544
weighted avg       0.36      0.37      0.33       544





f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

=== HistGradientBoosting ===
Accuracy: 0.4651
Macro F1:  0.3402

Classification report:

              precision    recall  f1-score   support

         ang       0.46      0.77      0.58        65
         exc       0.50      0.15      0.23        93
         fru       0.36      0.52      0.42       133
         hap       0.00      0.00      0.00        36
         neu       0.46      0.56      0.50       117
         sad       0.75      0.57      0.65        97
         sur       0.00      0.00      0.00         3

    accuracy                           0.47       544
   macro avg       0.36      0.37      0.34       544
weighted avg       0.46      0.47      0.44       544





f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

In [ ]:
# Cross-validation on the TRAIN split only (sessions 1-4)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cross_validate_model(name, model):
    cv_acc = []
    cv_f1 = []
    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr = X_train.iloc[train_idx]
        y_tr = y_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_val = y_train.iloc[val_idx]
        model.fit(X_tr, y_tr)
        y_val_pred = model.predict(X_val)
        cv_acc.append(accuracy_score(y_val, y_val_pred))
        cv_f1.append(f1_score(y_val, y_val_pred, average="macro"))
    print(f"=== {name} (CV) ===")
    print(f"CV Accuracy (mean±std): {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
    print(f"CV Macro F1 (mean±std):  {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")
    print("\n")

for name, model in models.items():
    cross_validate_model(name, model)